In [ ]:
#loading the dataset
from music21 import corpus
import pandas as pd
import re
from pandas import Series

scores4 = []
# όλα τα κομμάτια του Bach στο ενσωματωμένο corpus
for chorale in corpus.chorales.Iterator():
    if len(chorale.parts) == 4:          # το φίλτρο σου, όπως το ξέρεις
        scores4.append(chorale)

score_series = Series(scores4)
score_series = score_series.apply(lambda s: s.metadata.corpusFilePath)
# (γ) Πέρασμα 1: διπλά σε επίπεδο ολόκληρου path
ids = score_series.str.extract(r'bwv([\d.]+)', expand=False)
variants = score_series[ids.duplicated(keep=False)].sort_values()
print(variants)

variants = variants.drop_duplicates()
print(variants)

114       bach/bwv103.6.mxl
331       bach/bwv103.6.mxl
315       bach/bwv104.6.mxl
119       bach/bwv104.6.mxl
171       bach/bwv122.6.mxl
48        bach/bwv122.6.mxl
120     bach/bwv18.5-lz.mxl
95       bach/bwv18.5-w.mxl
88       bach/bwv194.12.mxl
250      bach/bwv194.12.mxl
59        bach/bwv194.6.mxl
249       bach/bwv194.6.mxl
8      bach/bwv248.12-2.mxl
342    bach/bwv248.12-2.mxl
274        bach/bwv25.6.mxl
247        bach/bwv25.6.mxl
300         bach/bwv267.mxl
4           bach/bwv267.mxl
21         bach/bwv28.6.mxl
83         bach/bwv28.6.mxl
191         bach/bwv283.mxl
298         bach/bwv283.mxl
149         bach/bwv3.6.mxl
299         bach/bwv3.6.mxl
309         bach/bwv325.mxl
228         bach/bwv325.mxl
229         bach/bwv335.mxl
287         bach/bwv335.mxl
308         bach/bwv339.mxl
138         bach/bwv339.mxl
192         bach/bwv343.mxl
293         bach/bwv343.mxl
188      bach/bwv36.4-2.mxl
81       bach/bwv36.4-2.mxl
296      bach/bwv36.4-2.mxl
125         bach/bwv

In [ ]:
paths = []
for chorale in corpus.chorales.Iterator():
    paths.append(chorale.metadata.corpusFilePath)

print(len(paths), len(set(paths)), len(variants))

371 349 22


In [ ]:
ids = variants.str.extract(r'bwv([\d.]+)', expand=False)
difharmon = variants[ids.duplicated(keep=False)].sort_values()
print(difharmon)

120    bach/bwv18.5-lz.mxl
95      bach/bwv18.5-w.mxl
dtype: object


In [ ]:
k = scores4[0].analyze('key')
print(k, '|', k.mode, '|', k.tonic)

G major | major | G


In [ ]:
p = scores4[0].parts[0]
print(p.partName)                          # ταυτότητα φωνής
print(p.getElementsByClass('Measure')[0].number) # πρώτο μέτρο: 0 ή 1; (auftakt test!)
p.measure(0).show('text')     # τι έχει μέσα το measure 0

Soprano
0
{0.0} <music21.layout.SystemLayout>
{0.0} <music21.clef.TrebleClef>
{0.0} <music21.key.Key of G major>
{0.0} <music21.meter.TimeSignature 3/4>
{0.0} <music21.note.Note G>


In [ ]:
from operator import attrgetter
#Now we can start creating the dataframe, by using as
#references to explore the chorales list and create a data frame,
#the corale's name and the chorale's path

#First we will save the name of the

measure_names_data = {"vector_name": ["bar", "suffix", "offset", "len",
                    "anacrusis"],
                      "measure_name": ["number", "numberSuffix", "offset",
                     "duration.quarterLength", "paddingLeft"]}

data = {"bar": [],
        "suffix": [],
        "offset": [],
        "len": [],
        "anacrusis": []}
for m in p.getElementsByClass('Measure'):
    for mn in measure_names_data["measure_name"]:
      vector_name_index = measure_names_data["measure_name"].index(mn)
      data[measure_names_data["vector_name"][vector_name_index]].append(attrgetter(mn)(m))
    print("Bar", f"{m.number}",
          "Suffix:", m.numberSuffix,
          "offset:", m.offset,
          "len:", m.duration.quarterLength,
          "anacrusis:", m.paddingLeft,
          "notes:", [(n.offset, n.pitch.midi, n.duration.quarterLength) for n in m.notes])

#Creating the first part of the dataset
chorales = pd.DataFrame(data)
#in order to be able to handle the data later
chorales["suffix"] = chorales["suffix"].fillna('')

Bar 0 Suffix: None offset: 0.0 len: 1.0 anacrusis: 2.0 notes: [(0.0, 67, 1.0)]
Bar 1 Suffix: None offset: 1.0 len: 3.0 anacrusis: 0.0 notes: [(0.0, 67, 2.0), (2.0, 74, 1.0)]
Bar 2 Suffix: None offset: 4.0 len: 3.0 anacrusis: 0.0 notes: [(0.0, 71, 1.5), (1.5, 69, 0.5), (2.0, 67, 1.0)]
Bar 3 Suffix: None offset: 7.0 len: 3.0 anacrusis: 0.0 notes: [(0.0, 67, 1.5), (1.5, 69, 0.5), (2.0, 71, 1.0)]
Bar 4 Suffix: None offset: 10.0 len: 3.0 anacrusis: 0.0 notes: [(0.0, 69, 2.0), (2.0, 71, 1.0)]
Bar 5 Suffix: None offset: 13.0 len: 3.0 anacrusis: 0.0 notes: [(0.0, 74, 2.0), (2.0, 72, 1.0)]
Bar 6 Suffix: None offset: 16.0 len: 3.0 anacrusis: 0.0 notes: [(0.0, 71, 1.0), (1.0, 69, 2.0)]
Bar 7 Suffix: None offset: 19.0 len: 2.0 anacrusis: 0.0 notes: [(0.0, 67, 2.0)]
Bar 7 Suffix: a offset: 21.0 len: 1.0 anacrusis: 2.0 notes: [(0.0, 71, 1.0)]
Bar 8 Suffix: None offset: 22.0 len: 3.0 anacrusis: 0.0 notes: [(0.0, 71, 1.0), (1.0, 72, 1.0), (2.0, 74, 1.0)]
Bar 9 Suffix: None offset: 25.0 len: 3.0 anacru

In [ ]:
#Example of organised voice data, easier to work with and explore

for j in chorales["bar"].unique():
  for suffix in chorales.loc[chorales["bar"] == j, "suffix"]:

    if suffix != None:
      print("current bar:", j, suffix)
      barid = f"{j}{suffix}"
    else:
      print("current bar:", j)
    for i in range(len(scores4[0].parts)):
      p = scores4[0].parts[i]
      print(p.partName)
      display = [disp for disp in p.getElementsByClass('Measure')
        if disp.number == j and (disp.numberSuffix or '') == suffix][0]
      display.show('text')


current bar: 0 
Soprano
{0.0} <music21.layout.SystemLayout>
{0.0} <music21.clef.TrebleClef>
{0.0} <music21.key.Key of G major>
{0.0} <music21.meter.TimeSignature 3/4>
{0.0} <music21.note.Note G>
Alto
{0.0} <music21.layout.SystemLayout>
{0.0} <music21.layout.StaffLayout distance 130, staffNumber 1, staffSize None, staffLines None>
{0.0} <music21.clef.TrebleClef>
{0.0} <music21.key.Key of G major>
{0.0} <music21.meter.TimeSignature 3/4>
{0.0} <music21.note.Note D>
Tenor
{0.0} <music21.layout.SystemLayout>
{0.0} <music21.layout.StaffLayout distance 130, staffNumber 1, staffSize None, staffLines None>
{0.0} <music21.clef.BassClef>
{0.0} <music21.key.Key of G major>
{0.0} <music21.meter.TimeSignature 3/4>
{0.0} <music21.note.Note B>
Bass
{0.0} <music21.layout.SystemLayout>
{0.0} <music21.layout.StaffLayout distance 130, staffNumber 1, staffSize None, staffLines None>
{0.0} <music21.clef.BassClef>
{0.0} <music21.key.Key of G major>
{0.0} <music21.meter.TimeSignature 3/4>
{0.0} <music21.note.

In [ ]:
data


{'bar': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21],
 'suffix': [None,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  'a',
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  'a',
  None,
  None,
  None,
  None,
  None,
  None,
  None],
 'offset': [0.0,
  1.0,
  4.0,
  7.0,
  10.0,
  13.0,
  16.0,
  19.0,
  21.0,
  22.0,
  25.0,
  28.0,
  31.0,
  34.0,
  37.0,
  40.0,
  42.0,
  43.0,
  46.0,
  49.0,
  52.0,
  55.0,
  58.0,
  61.0],
 'len': [1.0,
  3.0,
  3.0,
  3.0,
  3.0,
  3.0,
  3.0,
  2.0,
  1.0,
  3.0,
  3.0,
  3.0,
  3.0,
  3.0,
  3.0,
  2.0,
  1.0,
  3.0,
  3.0,
  3.0,
  3.0,
  3.0,
  3.0,
  2.0],
 'anacrusis': [2.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  2.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  2.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0]}